# Walmart Feature Engineering

## Milestone 2.2 — Feature Engineering

This notebook creates predictive features from historical Walmart sales, pricing, calendar, weather, and economic data.

The engineered features will later be used to train demand forecasting models.

In [33]:
import pandas as pd
import numpy as np

In [34]:
from sqlalchemy import create_engine
import os
from dotenv import load_dotenv

load_dotenv("../.env")

connection_string = (
    f"postgresql+psycopg2://"
    f"{os.getenv('POSTGRES_USER')}:"
    f"{os.getenv('POSTGRES_PASSWORD')}@"
    f"{os.getenv('POSTGRES_HOST')}:"
    f"{os.getenv('POSTGRES_PORT')}/"
    f"{os.getenv('POSTGRES_DB')}"
)

engine = create_engine(connection_string)

In [35]:
query = """
SELECT
    s.date,
    s.item_id,
    s.store_id,
    s.units_sold
FROM fact_sales AS s
WHERE s.store_id = 'CA_1'
ORDER BY
    s.item_id,
    s.date;
"""

df = pd.read_sql(query, engine)

In [36]:
df.head()

,date,item_id,store_id,units_sold
0,2011-01-29,FOODS_1_001,CA_1,3
1,2011-01-30,FOODS_1_001,CA_1,0
2,2011-01-31,FOODS_1_001,CA_1,0
3,2011-02-01,FOODS_1_001,CA_1,1
4,2011-02-02,FOODS_1_001,CA_1,4


In [37]:
df.shape

(5918109, 4)

## Lag Features

Lag features capture historical demand for each product-store combination.

For each item, the following features are created:

- `lag_1` — units sold 1 day earlier
- `lag_7` — units sold 7 days earlier
- `lag_30` — units sold 30 days earlier

These features allow forecasting models to learn short-term, weekly, and longer-term demand patterns.

The first observations for each product contain missing lag values because there is not enough historical data available yet.

In [38]:
df["lag_1"] = (
    df.groupby(["store_id", "item_id"])["units_sold"]
    .shift(1)
)

df["lag_7"] = (
    df.groupby(["store_id", "item_id"])["units_sold"]
    .shift(7)
)

df["lag_30"] = (
    df.groupby(["store_id", "item_id"])["units_sold"]
    .shift(30)
)

In [39]:
df[
    [
        "date",
        "item_id",
        "units_sold",
        "lag_1",
        "lag_7",
        "lag_30",
    ]
].head(35)

,date,item_id,units_sold,lag_1,lag_7,lag_30
0,2011-01-29,FOODS_1_001,3,NaN,NaN,NaN
1,2011-01-30,FOODS_1_001,0,3.0,NaN,NaN
2,2011-01-31,FOODS_1_001,0,0.0,NaN,NaN
3,2011-02-01,FOODS_1_001,1,0.0,NaN,NaN
4,2011-02-02,FOODS_1_001,4,1.0,NaN,NaN
5,2011-02-03,FOODS_1_001,2,4.0,NaN,NaN
6,2011-02-04,FOODS_1_001,0,2.0,NaN,NaN
7,2011-02-05,FOODS_1_001,2,0.0,3.0,NaN
8,2011-02-06,FOODS_1_001,0,2.0,0.0,NaN
9,2011-02-07,FOODS_1_001,0,0.0,0.0,NaN


### Observation

The lag features were generated correctly for each product independently.

For example, `lag_1` reflects the previous day's sales, while `lag_7` reflects sales from the same product seven days earlier.

Missing values at the beginning of each product's history are expected and will be handled before model training.

## Rolling Average Features

Rolling averages smooth short-term fluctuations in demand and help capture recent sales trends.

The rolling calculations use only previous sales observations so that future information is not leaked into the model.

In [40]:
df["rolling_avg_7"] = (
    df.groupby(["store_id", "item_id"])["units_sold"]
    .transform(
        lambda x: x.shift(1).rolling(
            window=7,
            min_periods=1
        ).mean()
    )
)

df["rolling_avg_28"] = (
    df.groupby(["store_id", "item_id"])["units_sold"]
    .transform(
        lambda x: x.shift(1).rolling(
            window=28,
            min_periods=1
        ).mean()
    )
)

In [41]:
df[
    [
        "date",
        "item_id",
        "units_sold",
        "lag_1",
        "lag_7",
        "lag_30",
        "rolling_avg_7",
        "rolling_avg_28",
    ]
].head(35)

,date,item_id,units_sold,lag_1,lag_7,lag_30,rolling_avg_7,rolling_avg_28
0,2011-01-29,FOODS_1_001,3,NaN,NaN,NaN,NaN,NaN
1,2011-01-30,FOODS_1_001,0,3.0,NaN,NaN,3.000000,3.000000
2,2011-01-31,FOODS_1_001,0,0.0,NaN,NaN,1.500000,1.500000
3,2011-02-01,FOODS_1_001,1,0.0,NaN,NaN,1.000000,1.000000
4,2011-02-02,FOODS_1_001,4,1.0,NaN,NaN,1.000000,1.000000
5,2011-02-03,FOODS_1_001,2,4.0,NaN,NaN,1.600000,1.600000
6,2011-02-04,FOODS_1_001,0,2.0,NaN,NaN,1.666667,1.666667
7,2011-02-05,FOODS_1_001,2,0.0,3.0,NaN,1.428571,1.428571
8,2011-02-06,FOODS_1_001,0,2.0,0.0,NaN,1.285714,1.500000
9,2011-02-07,FOODS_1_001,0,0.0,0.0,NaN,1.285714,1.333333


## Rolling Average Features

Rolling averages capture recent demand trends while smoothing daily fluctuations.

- `rolling_avg_7` represents the average sales over the previous 7 days.
- `rolling_avg_28` represents the average sales over the previous 28 days.

The current day's sales are excluded using a one-day shift to prevent data leakage.

## Calendar Features

Calendar features capture seasonal and time-based patterns in product demand.

The following features are extracted from the sales date:

- `day_of_week` — day of the week (Monday = 0, Sunday = 6)
- `month` — month of the year
- `year` — calendar year
- `day_of_month` — day within the month
- `is_weekend` — indicates whether the date falls on Saturday or Sunday

These features help forecasting models learn recurring weekly and seasonal demand patterns.

In [42]:
df["date"] = pd.to_datetime(df["date"])

df["day_of_week"] = df["date"].dt.dayofweek
df["month"] = df["date"].dt.month
df["year"] = df["date"].dt.year
df["day_of_month"] = df["date"].dt.day

df["is_weekend"] = (
    df["day_of_week"]
    .isin([5, 6])
    .astype(int)
)

In [43]:
df[
    [
        "date",
        "item_id",
        "units_sold",
        "day_of_week",
        "month",
        "year",
        "day_of_month",
        "is_weekend",
    ]
].head(15)

,date,item_id,units_sold,day_of_week,month,year,day_of_month,is_weekend
0,2011-01-29,FOODS_1_001,3,5,1,2011,29,1
1,2011-01-30,FOODS_1_001,0,6,1,2011,30,1
2,2011-01-31,FOODS_1_001,0,0,1,2011,31,0
3,2011-02-01,FOODS_1_001,1,1,2,2011,1,0
4,2011-02-02,FOODS_1_001,4,2,2,2011,2,0
5,2011-02-03,FOODS_1_001,2,3,2,2011,3,0
6,2011-02-04,FOODS_1_001,0,4,2,2011,4,0
7,2011-02-05,FOODS_1_001,2,5,2,2011,5,1
8,2011-02-06,FOODS_1_001,0,6,2,2011,6,1
9,2011-02-07,FOODS_1_001,0,0,2,2011,7,0


### Observation

The calendar features were successfully extracted from the sales date.

Weekend dates are correctly identified using `is_weekend`, while the day, month, and year variables provide additional time-based information that can help the forecasting model identify weekly and seasonal demand patterns.

## Demand Change Features

Demand change features measure how recent sales differ from previous sales levels.

These features help the forecasting model identify whether product demand is increasing, decreasing, or remaining relatively stable over time.

In [44]:
df["sales_change_1"] = df["lag_1"] - df["lag_7"]

df["sales_change_7_28"] = (
    df["rolling_avg_7"] - df["rolling_avg_28"]
)

In [45]:
df[
    [
        "date",
        "item_id",
        "units_sold",
        "lag_1",
        "lag_7",
        "rolling_avg_7",
        "rolling_avg_28",
        "sales_change_1",
        "sales_change_7_28",
    ]
].head(35)

,date,item_id,units_sold,lag_1,lag_7,rolling_avg_7,rolling_avg_28,sales_change_1,sales_change_7_28
0,2011-01-29,FOODS_1_001,3,NaN,NaN,NaN,NaN,NaN,NaN
1,2011-01-30,FOODS_1_001,0,3.0,NaN,3.000000,3.000000,NaN,0.000000
2,2011-01-31,FOODS_1_001,0,0.0,NaN,1.500000,1.500000,NaN,0.000000
3,2011-02-01,FOODS_1_001,1,0.0,NaN,1.000000,1.000000,NaN,0.000000
4,2011-02-02,FOODS_1_001,4,1.0,NaN,1.000000,1.000000,NaN,0.000000
5,2011-02-03,FOODS_1_001,2,4.0,NaN,1.600000,1.600000,NaN,0.000000
6,2011-02-04,FOODS_1_001,0,2.0,NaN,1.666667,1.666667,NaN,0.000000
7,2011-02-05,FOODS_1_001,2,0.0,3.0,1.428571,1.428571,-3.0,0.000000
8,2011-02-06,FOODS_1_001,0,2.0,0.0,1.285714,1.500000,2.0,-0.214286
9,2011-02-07,FOODS_1_001,0,0.0,0.0,1.285714,1.333333,0.0,-0.047619


## Price Features

Product prices can influence customer demand, so historical pricing information is added to the forecasting dataset.

Price features will allow the forecasting model to learn relationships between selling price and units sold.

In [46]:
price_query = """
SELECT *
FROM fact_prices
LIMIT 5;
"""

prices = pd.read_sql(price_query, engine)

prices

,store_id,item_id,wm_yr_wk,sell_price
0,CA_1,HOBBIES_1_001,11325,9.58
1,CA_1,HOBBIES_1_001,11326,9.58
2,CA_1,HOBBIES_1_001,11327,8.26
3,CA_1,HOBBIES_1_001,11328,8.26
4,CA_1,HOBBIES_1_001,11329,8.26


### Price-to-Date Mapping

Price data is recorded by Walmart week (`wm_yr_wk`) rather than individual date.

The calendar table will be used to map each sales date to its corresponding Walmart week before price information is joined to the sales dataset.

In [47]:
calendar_query = """
SELECT
    date,
    wm_yr_wk
FROM dim_calendar
ORDER BY date
LIMIT 10;
"""

calendar = pd.read_sql(calendar_query, engine)

calendar

,date,wm_yr_wk
0,2011-01-29,11101
1,2011-01-30,11101
2,2011-01-31,11101
3,2011-02-01,11101
4,2011-02-02,11101
5,2011-02-03,11101
6,2011-02-04,11101
7,2011-02-05,11102
8,2011-02-06,11102
9,2011-02-07,11102


## Merge Price Features

Daily sales records are linked to weekly Walmart prices using the calendar table.

The merge happens in two steps:

1. Map each sales date to its Walmart week (`wm_yr_wk`)
2. Join weekly product prices using `store_id`, `item_id`, and `wm_yr_wk`

This adds `sell_price` to each daily sales observation.

In [48]:
# Load full calendar mapping
calendar_map = pd.read_sql(
    """
    SELECT
        date,
        wm_yr_wk
    FROM dim_calendar
    """,
    engine
)

calendar_map["date"] = pd.to_datetime(calendar_map["date"])

# Add Walmart week to sales data
df = df.merge(
    calendar_map,
    on="date",
    how="left",
    validate="many_to_one"
)

In [49]:
prices_ca1 = pd.read_sql(
    """
    SELECT
        store_id,
        item_id,
        wm_yr_wk,
        sell_price
    FROM fact_prices
    WHERE store_id = 'CA_1'
    """,
    engine
)

In [50]:
df = df.merge(
    prices_ca1,
    on=[
        "store_id",
        "item_id",
        "wm_yr_wk",
    ],
    how="left",
    validate="many_to_one"
)

In [51]:
print(df.shape)

df[
    [
        "date",
        "item_id",
        "store_id",
        "wm_yr_wk",
        "units_sold",
        "sell_price",
    ]
].head(20)

(5918109, 18)


,date,item_id,store_id,wm_yr_wk,units_sold,sell_price
0,2011-01-29,FOODS_1_001,CA_1,11101,3,2.0
1,2011-01-30,FOODS_1_001,CA_1,11101,0,2.0
2,2011-01-31,FOODS_1_001,CA_1,11101,0,2.0
3,2011-02-01,FOODS_1_001,CA_1,11101,1,2.0
4,2011-02-02,FOODS_1_001,CA_1,11101,4,2.0
5,2011-02-03,FOODS_1_001,CA_1,11101,2,2.0
6,2011-02-04,FOODS_1_001,CA_1,11101,0,2.0
7,2011-02-05,FOODS_1_001,CA_1,11102,2,2.0
8,2011-02-06,FOODS_1_001,CA_1,11102,0,2.0
9,2011-02-07,FOODS_1_001,CA_1,11102,0,2.0


## Price Change Features

Price-change features capture how a product's current selling price compares with its recent historical price.

These features help the forecasting model learn whether changes in price are associated with changes in demand.

In [52]:
df["price_lag_1"] = (
    df.groupby(["store_id", "item_id"])["sell_price"]
    .shift(1)
)

df["price_change_1"] = (
    df["sell_price"] - df["price_lag_1"]
)

df["price_pct_change_1"] = (
    df["price_change_1"] / df["price_lag_1"]
)

In [53]:
df[
    [
        "date",
        "item_id",
        "sell_price",
        "price_lag_1",
        "price_change_1",
        "price_pct_change_1",
    ]
].head(40)

,date,item_id,sell_price,price_lag_1,price_change_1,price_pct_change_1
0,2011-01-29,FOODS_1_001,2.0,NaN,NaN,NaN
1,2011-01-30,FOODS_1_001,2.0,2.0,0.0,0.0
2,2011-01-31,FOODS_1_001,2.0,2.0,0.0,0.0
3,2011-02-01,FOODS_1_001,2.0,2.0,0.0,0.0
4,2011-02-02,FOODS_1_001,2.0,2.0,0.0,0.0
5,2011-02-03,FOODS_1_001,2.0,2.0,0.0,0.0
6,2011-02-04,FOODS_1_001,2.0,2.0,0.0,0.0
7,2011-02-05,FOODS_1_001,2.0,2.0,0.0,0.0
8,2011-02-06,FOODS_1_001,2.0,2.0,0.0,0.0
9,2011-02-07,FOODS_1_001,2.0,2.0,0.0,0.0


### Observation

The price features were generated correctly.

Most daily observations show no price change because Walmart prices are recorded at the weekly level. Therefore, consecutive days within the same Walmart week typically share the same selling price.

The first observation for each product contains missing lag values because no previous price is available.

## Holiday and SNAP Features

Holiday, event, and SNAP indicators can influence consumer purchasing behavior.

The calendar table provides:

- Holiday/event indicators
- Event categories
- SNAP participation flags

These features help the forecasting model capture demand changes associated with special events and benefit-payment periods.

In [55]:
calendar_features = pd.read_sql(
    """
    SELECT
        date,
        event_name_1,
        event_type_1,
        event_name_2,
        event_type_2,
        snap_ca,
        snap_tx,
        snap_wi
    FROM dim_calendar
    """,
    engine
)

calendar_features["date"] = pd.to_datetime(
    calendar_features["date"]
)

In [56]:
df = df.merge(
    calendar_features,
    on="date",
    how="left",
    validate="many_to_one"
)

In [57]:
df["is_event"] = (
    df["event_name_1"].notna()
    | df["event_name_2"].notna()
).astype(int)

df["snap_active"] = df["snap_ca"].astype(int)

In [58]:
df[
    [
        "date",
        "item_id",
        "units_sold",
        "event_name_1",
        "event_type_1",
        "is_event",
        "snap_active",
    ]
].head(40)

,date,item_id,units_sold,event_name_1,event_type_1,is_event,snap_active
0,2011-01-29,FOODS_1_001,3,NaN,NaN,0,0
1,2011-01-30,FOODS_1_001,0,NaN,NaN,0,0
2,2011-01-31,FOODS_1_001,0,NaN,NaN,0,0
3,2011-02-01,FOODS_1_001,1,NaN,NaN,0,1
4,2011-02-02,FOODS_1_001,4,NaN,NaN,0,1
5,2011-02-03,FOODS_1_001,2,NaN,NaN,0,1
6,2011-02-04,FOODS_1_001,0,NaN,NaN,0,1
7,2011-02-05,FOODS_1_001,2,NaN,NaN,0,1
8,2011-02-06,FOODS_1_001,0,SuperBowl,Sporting,1,1
9,2011-02-07,FOODS_1_001,0,NaN,NaN,0,1


### Observation

Holiday/event and SNAP features were successfully added to the forecasting dataset.

Special events such as the Super Bowl, Valentine's Day, Presidents' Day, and LentStart are correctly identified by `is_event`.

The `snap_active` feature also captures California SNAP-active periods, providing additional information that may explain changes in consumer demand.

## Weather Features

Weather conditions may influence retail demand and customer purchasing behavior.

Weather data will be joined to the sales dataset by date and store location to provide additional external predictors for demand forecasting.

In [59]:
weather_query = """
SELECT *
FROM fact_weather
LIMIT 10;
"""

weather = pd.read_sql(weather_query, engine)

weather

,date,state_id,temperature_max,temperature_min,precipitation,snowfall,wind_speed_max
0,2011-01-29,CA,18.1,6.1,0.0,0.0,10.9
1,2011-01-30,CA,13.1,7.6,1.0,0.0,16.8
2,2011-01-31,CA,18.9,4.3,0.0,0.0,11.0
3,2011-02-01,CA,17.2,4.9,0.0,0.0,10.9
4,2011-02-02,CA,16.4,2.1,0.0,0.0,8.8
5,2011-02-03,CA,17.9,1.6,0.0,0.0,12.9
6,2011-02-04,CA,20.7,3.0,0.0,0.0,10.7
7,2011-02-05,CA,21.8,6.6,0.0,0.0,9.4
8,2011-02-06,CA,25.6,7.8,0.0,0.0,11.7
9,2011-02-07,CA,24.3,8.5,0.0,0.0,11.0


### Merge Weather Data

Daily weather conditions are joined to the sales dataset using the sales date.

Because the current feature-engineering dataset represents store `CA_1`, California weather observations are used.

The weather variables include temperature, precipitation, snowfall, and wind speed.

In [60]:
weather_ca = pd.read_sql(
    """
    SELECT
        date,
        temperature_max,
        temperature_min,
        precipitation,
        snowfall,
        wind_speed_max
    FROM fact_weather
    WHERE state_id = 'CA'
    """,
    engine
)

weather_ca["date"] = pd.to_datetime(weather_ca["date"])

In [61]:
df = df.merge(
    weather_ca,
    on="date",
    how="left",
    validate="many_to_one"
)

In [62]:
print(df.shape)

(5918109, 35)


In [63]:
df[
    [
        "date",
        "item_id",
        "units_sold",
        "temperature_max",
        "temperature_min",
        "precipitation",
        "snowfall",
        "wind_speed_max",
    ]
].head(20)

,date,item_id,units_sold,temperature_max,temperature_min,precipitation,snowfall,wind_speed_max
0,2011-01-29,FOODS_1_001,3,18.1,6.1,0.0,0.0,10.9
1,2011-01-30,FOODS_1_001,0,13.1,7.6,1.0,0.0,16.8
2,2011-01-31,FOODS_1_001,0,18.9,4.3,0.0,0.0,11.0
3,2011-02-01,FOODS_1_001,1,17.2,4.9,0.0,0.0,10.9
4,2011-02-02,FOODS_1_001,4,16.4,2.1,0.0,0.0,8.8
5,2011-02-03,FOODS_1_001,2,17.9,1.6,0.0,0.0,12.9
6,2011-02-04,FOODS_1_001,0,20.7,3.0,0.0,0.0,10.7
7,2011-02-05,FOODS_1_001,2,21.8,6.6,0.0,0.0,9.4
8,2011-02-06,FOODS_1_001,0,25.6,7.8,0.0,0.0,11.7
9,2011-02-07,FOODS_1_001,0,24.3,8.5,0.0,0.0,11.0


## Economic Features

Economic indicators provide broader market context that may influence consumer demand.

The economic data stored in `dim_economic_series` will be inspected and then merged with the sales dataset where appropriate.

In [64]:
economic_query = """
SELECT *
FROM dim_economic_series
LIMIT 20;
"""

economic = pd.read_sql(economic_query, engine)

economic

,series_id,series_name,frequency,units
0,CPIAUCSL,Consumer Price Index,Monthly,Index
1,UNRATE,Unemployment Rate,Monthly,Percent
2,FEDFUNDS,Federal Funds Rate,Monthly,Percent


### Economic Indicator Values

The economic-series dimension identifies which indicators are available.

The corresponding historical observations are stored in `fact_economic_indicator` and will be joined to the sales dataset by observation date.

In [65]:
economic_values_query = """
SELECT *
FROM fact_economic_indicator
ORDER BY observation_date
LIMIT 20;
"""

economic_values = pd.read_sql(
    economic_values_query,
    engine
)

economic_values

,series_id,observation_date,value
0,CPIAUCSL,2011-01-01,221.187
1,UNRATE,2011-01-01,9.100
2,FEDFUNDS,2011-01-01,0.170
3,CPIAUCSL,2011-02-01,221.898
4,UNRATE,2011-02-01,9.000
5,FEDFUNDS,2011-02-01,0.160
6,CPIAUCSL,2011-03-01,223.046
7,UNRATE,2011-03-01,9.000
8,FEDFUNDS,2011-03-01,0.140
9,CPIAUCSL,2011-04-01,224.093


### Reshape Economic Indicators

The FRED observations are stored in long format, with one row per economic series and observation date.

The data will be reshaped into a wide format so that each date contains separate columns for CPI, unemployment rate, and the federal funds rate.

In [66]:
economic_full = pd.read_sql(
    """
    SELECT
        series_id,
        observation_date,
        value
    FROM fact_economic_indicator
    ORDER BY observation_date
    """,
    engine
)

economic_full["observation_date"] = pd.to_datetime(
    economic_full["observation_date"]
)

economic_wide = (
    economic_full
    .pivot(
        index="observation_date",
        columns="series_id",
        values="value"
    )
    .reset_index()
)

economic_wide = economic_wide.rename(
    columns={
        "CPIAUCSL": "cpi",
        "UNRATE": "unemployment_rate",
        "FEDFUNDS": "federal_funds_rate",
    }
)

economic_wide.head()

series_id,observation_date,cpi,federal_funds_rate,unemployment_rate
0,2011-01-01,221.187,0.17,9.1
1,2011-02-01,221.898,0.16,9.0
2,2011-03-01,223.046,0.14,9.0
3,2011-04-01,224.093,0.10,9.1
4,2011-05-01,224.806,0.09,9.0


### Merge Economic Features

Monthly economic indicators are aligned with daily sales observations.

Each sales date receives the most recent available values for:

- Consumer Price Index (`cpi`)
- Unemployment Rate (`unemployment_rate`)
- Federal Funds Rate (`federal_funds_rate`)

This provides broader economic context for daily demand forecasting.

In [67]:
df = df.sort_values("date").copy()
economic_wide = economic_wide.sort_values("observation_date").copy()

df = pd.merge_asof(
    df,
    economic_wide,
    left_on="date",
    right_on="observation_date",
    direction="backward"
)

In [68]:
df = df.drop(
    columns=["observation_date"]
)

In [69]:
print(df.shape)

df[
    [
        "date",
        "item_id",
        "units_sold",
        "cpi",
        "unemployment_rate",
        "federal_funds_rate",
    ]
].head(20)

(5918109, 38)


,date,item_id,units_sold,cpi,unemployment_rate,federal_funds_rate
0,2011-01-29,FOODS_1_001,3,221.187,9.1,0.17
1,2011-01-29,HOBBIES_2_100,0,221.187,9.1,0.17
2,2011-01-29,HOUSEHOLD_1_076,0,221.187,9.1,0.17
3,2011-01-29,FOODS_3_697,8,221.187,9.1,0.17
4,2011-01-29,HOBBIES_1_312,5,221.187,9.1,0.17
5,2011-01-29,HOUSEHOLD_1_466,0,221.187,9.1,0.17
6,2011-01-29,FOODS_2_377,0,221.187,9.1,0.17
7,2011-01-29,HOUSEHOLD_1_171,0,221.187,9.1,0.17
8,2011-01-29,FOODS_3_485,7,221.187,9.1,0.17
9,2011-01-29,FOODS_3_269,3,221.187,9.1,0.17


## Final Feature Cleanup

The engineered dataset now contains historical demand, pricing, calendar, event, SNAP, weather, and economic features.

Before model training, the dataset is cleaned by:

- Restoring product-date ordering
- Inspecting missing values
- Removing rows without sufficient historical lag information
- Selecting the final modeling features
- Validating the completed feature dataset

In [70]:
df = df.sort_values(
    ["store_id", "item_id", "date"]
).reset_index(drop=True)

In [71]:
missing_summary = (
    df.isnull()
    .sum()
    .sort_values(ascending=False)
)

missing_summary[
    missing_summary > 0
]

event_name_2          5905913
event_type_2          5905913
event_type_1          5436367
event_name_1          5436367
price_change_1        1132891
price_pct_change_1    1132891
price_lag_1           1132891
sell_price            1129842
lag_30                  91470
sales_change_1          21343
lag_7                   21343
lag_1                    3049
sales_change_7_28        3049
rolling_avg_7            3049
rolling_avg_28           3049
dtype: int64

### Missing Value Treatment

Missing values are handled according to their business meaning.

- Missing event names/types represent regular non-event days and are retained.
- Missing lag values occur at the beginning of each product's history.
- Rows without a 30-day demand history are removed before modeling.
- Missing selling prices represent periods where no historical price was available.
- Initial price-change values are treated as no observed price change.

In [72]:
# Remove rows without enough historical demand
df = df.dropna(
    subset=[
        "lag_30",
        "sell_price",
    ]
).copy()

# If no previous observed price exists,
# treat the current price as the initial reference price
df["price_lag_1"] = df["price_lag_1"].fillna(
    df["sell_price"]
)

df["price_change_1"] = df["price_change_1"].fillna(0)

df["price_pct_change_1"] = df[
    "price_pct_change_1"
].fillna(0)

In [73]:
print(df.shape)

(4748523, 38)


In [74]:
model_check_columns = [
    "units_sold",
    "lag_1",
    "lag_7",
    "lag_30",
    "rolling_avg_7",
    "rolling_avg_28",
    "sales_change_1",
    "sales_change_7_28",
    "sell_price",
    "price_lag_1",
    "price_change_1",
    "price_pct_change_1",
    "day_of_week",
    "month",
    "year",
    "day_of_month",
    "is_weekend",
    "is_event",
    "snap_active",
    "temperature_max",
    "temperature_min",
    "precipitation",
    "snowfall",
    "wind_speed_max",
    "cpi",
    "unemployment_rate",
    "federal_funds_rate",
]

df[model_check_columns].isnull().sum()

units_sold            0
lag_1                 0
lag_7                 0
lag_30                0
rolling_avg_7         0
rolling_avg_28        0
sales_change_1        0
sales_change_7_28     0
sell_price            0
price_lag_1           0
price_change_1        0
price_pct_change_1    0
day_of_week           0
month                 0
year                  0
day_of_month          0
is_weekend            0
is_event              0
snap_active           0
temperature_max       0
temperature_min       0
precipitation         0
snowfall              0
wind_speed_max        0
cpi                   0
unemployment_rate     0
federal_funds_rate    0
dtype: int64

## Final Modeling Dataset

The completed feature-engineering pipeline combines historical demand, pricing, calendar, event, SNAP, weather, and macroeconomic information.

The final dataset contains only the identifiers, target variable, and engineered features required for demand forecasting.

In [75]:
final_columns = [
    "date",
    "store_id",
    "item_id",
    "units_sold",

    # Demand history
    "lag_1",
    "lag_7",
    "lag_30",
    "rolling_avg_7",
    "rolling_avg_28",
    "sales_change_1",
    "sales_change_7_28",

    # Price
    "sell_price",
    "price_lag_1",
    "price_change_1",
    "price_pct_change_1",

    # Calendar
    "day_of_week",
    "month",
    "year",
    "day_of_month",
    "is_weekend",

    # Events / SNAP
    "is_event",
    "snap_active",

    # Weather
    "temperature_max",
    "temperature_min",
    "precipitation",
    "snowfall",
    "wind_speed_max",

    # Economic
    "cpi",
    "unemployment_rate",
    "federal_funds_rate",
]

model_df = df[final_columns].copy()

print("Final dataset shape:", model_df.shape)
print("Missing values:", model_df.isnull().sum().sum())

model_df.head()

Final dataset shape: (4748523, 30)
Missing values: 0


,date,store_id,item_id,units_sold,lag_1,lag_7,lag_30,rolling_avg_7,rolling_avg_28,sales_change_1,...,is_event,snap_active,temperature_max,temperature_min,precipitation,snowfall,wind_speed_max,cpi,unemployment_rate,federal_funds_rate
30,2011-02-28,CA_1,FOODS_1_001,0,2.0,0.0,3.0,2.000000,1.428571,2.0,...,0,0,16.8,0.9,0.0,0.0,12.1,221.898,9.0,0.16
31,2011-03-01,CA_1,FOODS_1_001,2,0.0,2.0,0.0,2.000000,1.428571,-2.0,...,0,1,16.3,3.0,0.0,0.0,12.5,223.046,9.0,0.14
32,2011-03-02,CA_1,FOODS_1_001,1,2.0,2.0,0.0,2.000000,1.464286,0.0,...,0,1,15.1,4.7,0.4,0.0,7.6,223.046,9.0,0.14
33,2011-03-03,CA_1,FOODS_1_001,7,1.0,2.0,1.0,1.857143,1.357143,-1.0,...,0,1,18.7,9.9,0.0,0.0,11.3,223.046,9.0,0.14
34,2011-03-04,CA_1,FOODS_1_001,1,7.0,4.0,4.0,2.571429,1.535714,3.0,...,0,1,23.1,6.9,0.0,0.0,8.0,223.046,9.0,0.14


## Save Engineered Dataset

The final feature-engineered dataset is saved for use in the forecasting and machine-learning pipeline.

This separates feature engineering from model training and allows future models to load a consistent, reproducible dataset.

In [80]:
from pathlib import Path

output_dir = Path("../data/processed")
output_dir.mkdir(parents=True, exist_ok=True)

output_path = output_dir / "walmart_features.pkl"

model_df.to_pickle(output_path)

print(f"Saved feature dataset to: {output_path}")
print(f"Rows: {len(model_df):,}")
print(f"Columns: {len(model_df.columns)}")

Saved feature dataset to: ..\data\processed\walmart_features.pkl
Rows: 4,748,523
Columns: 30
